# J-STAGE 論文ページからの参考文献リスト取得（Selenium版）

`requests` で取得した生HTMLには参考文献リスト（`<ul id="article-overview-references-list">`）が
含まれておらず、JavaScriptによる非同期レンダリング後に挿入されている可能性が高いことが分かったため、
実ブラウザを操作してレンダリング後のHTMLを取得する **Selenium** に切り替えます。

## 事前準備
- `pip install selenium`
- ローカルにGoogle Chrome（またはChromium）がインストールされていること
- Selenium 4.6以降であれば、`Selenium Manager` が対応するChromeDriverを自動取得するため、
  ChromeDriverを別途手動でダウンロードする必要は基本的にありません。

## 構成
- `create_driver()`：ヘッドレスChromeのdriverを作成
- `get_references(url, driver=None)`：単一URLから参考文献リストを取得（driverを渡さなければ内部で使い捨てのdriverを作成）
- `get_references_batch(urls)`：driverを使い回しながら複数URLをまとめて処理


In [ ]:
!pip install selenium beautifulsoup4 -q


In [ ]:
import re
import time
from typing import List, Dict, Optional

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36"
)


def create_driver(headless: bool = True) -> webdriver.Chrome:
    """
    ヘッドレスChromeのSelenium WebDriverを作成する。
    Selenium 4.6+ なら Selenium Manager が対応するChromeDriverを自動取得する。
    """
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1280,1800")
    options.add_argument(f"user-agent={USER_AGENT}")
    driver = webdriver.Chrome(options=options)
    return driver


## 参考文献抽出ロジック（HTML → リスト）

パース処理自体はrequests版と同じロジックです。取得元がSelenium(`driver.page_source`)に変わるだけです。


In [ ]:
def _parse_references(html: str, ul_id: str) -> List[str]:
    soup = BeautifulSoup(html, "html.parser")
    ul = soup.find("ul", id=ul_id)
    if ul is None:
        return []

    references = []
    for li in ul.find_all("li"):
        span = li.find("span", class_="reference-num-txt")
        if span is None:
            continue
        text = span.get_text(separator=" ", strip=True)
        text = re.sub(r"\s+", " ", text)
        references.append(text)
    return references


def get_references(
    url: str,
    driver: Optional[webdriver.Chrome] = None,
    ul_id: str = "article-overview-references-list",
    wait_sec: int = 20,
) -> List[str]:
    """
    J-STAGEの論文概要ページ(URL)をSeleniumでレンダリングし、参考文献リストを取得する。

    Parameters
    ----------
    url : str
        論文の概要ページURL
    driver : selenium.webdriver.Chrome, optional
        使い回したいdriverがあれば渡す（バッチ処理向け）。
        指定しない場合はこの関数内でdriverを作成し、終了時にquitする。
    ul_id : str
        参考文献リストを囲む<ul>のid
    wait_sec : int
        対象要素が描画されるまでの最大待機秒数

    Returns
    -------
    List[str]
        各参考文献のテキストを1件ずつ格納したリスト。
        タイムアウトした場合や該当<ul>が無い場合は空リストを返す。
    """
    own_driver = driver is None
    if own_driver:
        driver = create_driver()

    try:
        driver.get(url)
        try:
            WebDriverWait(driver, wait_sec).until(
                EC.presence_of_element_located((By.ID, ul_id))
            )
        except TimeoutException:
            return []
        return _parse_references(driver.page_source, ul_id)
    finally:
        if own_driver:
            driver.quit()


## 動作確認（単一URL）

まずはHTMLがちゃんと取れているか、取得元テキストの分量を確認してから、参考文献リストを抽出します。


In [ ]:
url = "https://www.jstage.jst.go.jp/article/jssej/48/4/48_399/_article/-char/ja"

driver = create_driver()
driver.get(url)

# 対象要素が描画されるまで待機（見つからなければ例外にせずFalseで確認できるようにする）
try:
    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.ID, "article-overview-references-list"))
    )
    found = True
except TimeoutException:
    found = False

print("target ul found:", found)
print("page_source length:", len(driver.page_source))


In [ ]:
refs = _parse_references(driver.page_source, "article-overview-references-list")
driver.quit()

print(f"{len(refs)} 件の参考文献を取得しました。\n")
for i, r in enumerate(refs, 1):
    print(f"[{i}] {r}")


## 複数URLをまとめて処理する関数

driverを1つ作って使い回すことで、毎回ブラウザを起動するオーバーヘッドを避けます。


In [ ]:
def get_references_batch(
    urls: List[str],
    ul_id: str = "article-overview-references-list",
    wait_sec: int = 20,
    sleep_sec: float = 1.0,
    headless: bool = True,
) -> Dict[str, List[str]]:
    """
    複数のJ-STAGE論文ページURLから、それぞれの参考文献リストをまとめて取得する。
    内部で1つのdriverを使い回し、各リクエストの間に sleep_sec 秒のインターバルを入れる。

    Returns
    -------
    Dict[str, List[str]]
        {url: [参考文献テキスト, ...]} の辞書。
        取得に失敗したURLは値としてエラーメッセージを1件だけ含むリストを格納する。
    """
    driver = create_driver(headless=headless)
    results: Dict[str, List[str]] = {}
    try:
        for i, url in enumerate(urls):
            try:
                results[url] = get_references(url, driver=driver, ul_id=ul_id, wait_sec=wait_sec)
            except Exception as e:
                results[url] = [f"[ERROR] 取得に失敗しました: {e}"]

            if i < len(urls) - 1:
                time.sleep(sleep_sec)
    finally:
        driver.quit()

    return results


In [ ]:
urls = [
    "https://www.jstage.jst.go.jp/article/jssej/48/4/48_399/_article/-char/ja",
    # 同様の構造を持つ他のJ-STAGE論文ページのURLをここに追加
]

results = get_references_batch(urls)

for url, refs in results.items():
    print(f"=== {url} ===")
    print(f"{len(refs)} 件\n")
    for i, r in enumerate(refs, 1):
        print(f"[{i}] {r}")
    print()


## （補足）CSV/テキストへの書き出し


In [ ]:
import csv

with open("references.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["url", "no", "reference"])
    for url, refs in results.items():
        for i, r in enumerate(refs, 1):
            writer.writerow([url, i, r])

print("references.csv に保存しました。")
